# 03 — Text Model Prototyping

Prototype and evaluate the text branch:
- DeBERTa-v3 encoder
- BiLSTM + Additive Attention
- Hyperparameter experimentation
- Attention weight visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from ml.text_pipeline.deberta_encoder import DeBERTaEncoder
from ml.text_pipeline.bilstm_attention import BiLSTMAttention
from ml.text_pipeline.tokenizer import get_tokenizer
from ml.text_pipeline.preprocessing import preprocess_text

# Initialize components
encoder = DeBERTaEncoder(freeze=True)
feature_extractor = BiLSTMAttention(input_size=768, hidden_size=128)
tokenizer = get_tokenizer('deberta-v3')

print(f'Encoder hidden size: {encoder.hidden_size}')
print(f'Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}')
print(f'Feature extractor parameters: {sum(p.numel() for p in feature_extractor.parameters()):,}')

In [ ]:
# Test forward pass
sample_text = preprocess_text('I feel so hopeless and tired of everything')
encoding = tokenizer(sample_text, return_tensors='pt', max_length=128,
                     padding='max_length', truncation=True)

with torch.no_grad():
    hidden = encoder(encoding['input_ids'], encoding['attention_mask'])
    embedding, attn_weights = feature_extractor(hidden, encoding['attention_mask'])

print(f'Hidden states: {hidden.shape}')
print(f'Embedding: {embedding.shape}')
print(f'Attention weights: {attn_weights.shape}')

In [ ]:
# Visualize attention weights
import matplotlib.pyplot as plt

tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'].squeeze().tolist())
weights = attn_weights.squeeze().tolist()

# Filter special tokens
filtered = [(t, w) for t, w in zip(tokens, weights)
            if t not in ('[CLS]', '[SEP]', '[PAD]', '<s>', '</s>')]
if filtered:
    tok, wt = zip(*filtered)
    plt.figure(figsize=(14, 3))
    plt.bar(range(len(tok)), wt, color='#14b8a6')
    plt.xticks(range(len(tok)), tok, rotation=45, ha='right', fontsize=8)
    plt.title('Token Attention Weights')
    plt.tight_layout()
    plt.show()

## Next Steps

- Run full training with `src/text_pipeline/train_text_model.py`
- Experiment with unfreezing encoder layers
- Compare attention strategies